In [2]:
import sys
import psycopg2

# Add root directory to sys.path to import config.py
sys.path.append('..')
from config import get_db_connection_params

# Establish database connection
try:
    params = get_db_connection_params()
    conn = psycopg2.connect(**params)
    cursor = conn.cursor()
    print("✅ Database connection successful!")
except Exception as e:
    print("❌ Database connection error:", e)

✅ Database connection successful!


In [3]:
# Fetch all public base tables
cursor.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public' 
      AND table_type = 'BASE TABLE';
""")
tables = cursor.fetchall()

print(f"Total tables found: {len(tables)}")

Total tables found: 11


In [4]:
# Structural Audit - Baseline record counts
print("📊 Structural Audit - Baseline Record Counts:")
print("-" * 50)

for table in tables:
    table_name = table[0]
    cursor.execute(f"SELECT COUNT(*) FROM {table_name};")
    count = cursor.fetchone()[0]
    print(f"• {table_name:<35} : {count:,} rows")

# Close cursor and connection
cursor.close()
conn.close()
print("\n🔒 Connection closed successfully.")

📊 Structural Audit - Baseline Record Counts:
--------------------------------------------------
• product_category_translation        : 73 rows
• customers                           : 99,441 rows
• sellers                             : 3,095 rows
• products                            : 32,951 rows
• geolocation                         : 14,786 rows
• orders                              : 99,441 rows
• order_items                         : 112,650 rows
• order_payments                      : 103,886 rows
• order_reviews                       : 99,224 rows
• marketing_qualified_leads           : 8,000 rows
• closed_deals                        : 842 rows

🔒 Connection closed successfully.


In [5]:
import csv

# Re-open connection since it was closed in the previous cell
params = get_db_connection_params()
conn = psycopg2.connect(**params)
cursor = conn.cursor()

# Define target columns to audit for null/missing values
audit_targets = [
    ("orders", "order_id"),
    ("orders", "order_delivered_customer_date"),
    ("orders", "order_estimated_delivery_date"),
    ("order_items", "price"),
    ("order_items", "freight_value"),
    ("products", "product_category_name"),
    ("customers", "customer_zip_code_prefix")
]

quality_audit_rows = [("table_name", "column_name", "null_count", "total_rows", "null_percentage")]
print("✅ Connection re-opened and audit targets are set.")

✅ Connection re-opened and audit targets are set.


In [6]:
print("🔍 Auditing Quality Defect Metrics:")
print("=" * 70)

for table_name, col_name in audit_targets:
    audit_query = f"""
        SELECT 
            '{table_name}' AS table_name,
            '{col_name}' AS column_name,
            COUNT(*) - COUNT({col_name}) AS null_count,
            COUNT(*) AS total_rows,
            ROUND((COUNT(*) - COUNT({col_name}))::DECIMAL / COUNT(*) * 100, 2) AS null_percentage
        FROM {table_name};
    """
    cursor.execute(audit_query)
    result = cursor.fetchone()
    quality_audit_rows.append(result)
    
    print(f"• Table: {result[0]:<12} | Column: {result[1]:<30} | Nulls: {result[2]:<6} ({result[4]}%)")

🔍 Auditing Quality Defect Metrics:
• Table: orders       | Column: order_id                       | Nulls: 0      (0.00%)
• Table: orders       | Column: order_delivered_customer_date  | Nulls: 2965   (2.98%)
• Table: orders       | Column: order_estimated_delivery_date  | Nulls: 0      (0.00%)
• Table: order_items  | Column: price                          | Nulls: 5674   (5.04%)
• Table: order_items  | Column: freight_value                  | Nulls: 5599   (4.97%)
• Table: products     | Column: product_category_name          | Nulls: 602    (1.83%)
• Table: customers    | Column: customer_zip_code_prefix       | Nulls: 4867   (4.89%)


In [7]:
output_audit_path = '../outputs/phase2_quality_audit.csv'

with open(output_audit_path, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerows(quality_audit_rows)

print(f"✅ Quality audit log exported to: {output_audit_path}")

✅ Quality audit log exported to: ../outputs/phase2_quality_audit.csv


In [8]:
# Run SQL descriptive statistics
eda_query = """
    SELECT 'order_items' AS table_name, 'price' AS metric_column, 
           MIN(price)::NUMERIC(10,2) AS min_val, 
           MAX(price)::NUMERIC(10,2) AS max_val, 
           AVG(price)::NUMERIC(10,2) AS avg_val, 
           STDDEV(price)::NUMERIC(10,2) AS stddev_val,
           COUNT(*) AS total_records
    FROM order_items
    UNION ALL
    SELECT 'order_items', 'freight_value', 
           MIN(freight_value)::NUMERIC(10,2), MAX(freight_value)::NUMERIC(10,2), 
           AVG(freight_value)::NUMERIC(10,2), STDDEV(freight_value)::NUMERIC(10,2), COUNT(*)
    FROM order_items
    UNION ALL
    SELECT 'order_payments', 'payment_value', 
           MIN(payment_value)::NUMERIC(10,2), MAX(payment_value)::NUMERIC(10,2), 
           AVG(payment_value)::NUMERIC(10,2), STDDEV(payment_value)::NUMERIC(10,2), COUNT(*)
    FROM order_payments;
"""

cursor.execute(eda_query)
eda_results = cursor.fetchall()

headers = ["table_name", "metric_column", "min_val", "max_val", "avg_val", "stddev_val", "total_records"]
eda_summary_rows = [headers] + eda_results

print("📊 Descriptive Statistics Summary:")
print("=" * 80)
for row in eda_summary_rows:
    print(row)

📊 Descriptive Statistics Summary:
['table_name', 'metric_column', 'min_val', 'max_val', 'avg_val', 'stddev_val', 'total_records']
('order_payments', 'payment_value', Decimal('0.00'), Decimal('282700.00'), Decimal('455.47'), Decimal('3750.09'), 103886)
('order_items', 'freight_value', Decimal('0.00'), Decimal('19384.00'), Decimal('61.35'), Decimal('359.88'), 112650)
('order_items', 'price', Decimal('0.85'), Decimal('459000.00'), Decimal('361.47'), Decimal('3231.03'), 112650)


In [9]:
output_eda_path = '../outputs/phase2_eda_summary.csv'

with open(output_eda_path, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerows(eda_summary_rows)

print(f"✅ EDA summary exported to: {output_eda_path}")

# Close DB connections safely at the end
cursor.close()
conn.close()
print("🔒 Database connection closed safely.")

✅ EDA summary exported to: ../outputs/phase2_eda_summary.csv
🔒 Database connection closed safely.
